In [2]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings

warnings.filterwarnings('ignore')

In [3]:
# Cell 2: Generate Simulated Market Data
np.random.seed(42)

# Generate 2 years of daily trading data
dates = pd.date_range('2024-01-01', '2025-12-31', freq='B')
returns = np.random.normal(0.0005, 0.015, len(dates))
price_path = 100 * np.exp(np.cumsum(returns))

# Store in a DataFrame
df = pd.DataFrame({
    'Date': dates, 
    'Close_Price': price_path
}).set_index('Date')

In [4]:
# Cell 3: Voilà Interactive Widget and Logic

# 1. Define the UI Widgets
style = {'description_width': 'initial'}
fast_sma_widget = widgets.IntSlider(
    value=20, min=5, max=50, step=1, 
    description='Fast SMA Window:', style=style
)
slow_sma_widget = widgets.IntSlider(
    value=50, min=20, max=200, step=5, 
    description='Slow SMA Window:', style=style
)

# Output widget to hold the plot
plot_output = widgets.Output()

# 2. Define the core logic and plotting function
def update_dashboard(change=None):
    with plot_output:
        clear_output(wait=True)
        
        # Fetch current widget values
        fast_window = fast_sma_widget.value
        slow_window = slow_sma_widget.value
        
        # Calculate SMAs based on user input
        data = df.copy()
        data['Fast_SMA'] = data['Close_Price'].rolling(window=fast_window).mean()
        data['Slow_SMA'] = data['Close_Price'].rolling(window=slow_window).mean()
        
        # Plotting
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Asset Price
        ax.plot(data.index, data['Close_Price'], label='Asset Price', color='black', alpha=0.4, linewidth=1)
        
        # Moving Averages
        ax.plot(data.index, data['Fast_SMA'], label=f'{fast_window}-Day Fast SMA', color='blue', linewidth=2)
        ax.plot(data.index, data['Slow_SMA'], label=f'{slow_window}-Day Slow SMA', color='orange', linewidth=2)
        
        # Formatting
        ax.set_title('Algorithmic Trading: Interactive SMA Crossover', fontsize=14, fontweight='bold')
        ax.set_ylabel('Price ($)')
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='upper left')
        
        plt.tight_layout()
        plt.show()

# 3. Bind the widgets to the update function
fast_sma_widget.observe(update_dashboard, names='value')
slow_sma_widget.observe(update_dashboard, names='value')

# 4. Render the dashboard layout
ui = widgets.VBox([
    widgets.HTML("<h2>Strategy Parameter Optimization</h2><p>Adjust the sliders to test different moving average crossover sensitivities.</p>"),
    widgets.HBox([fast_sma_widget, slow_sma_widget]),
    plot_output
])

# Display everything and trigger the first plot
display(ui)
update_dashboard()